# Collection A Processing: Split by Method (TMT, iTRAQ, Label-Free)

This notebook processes **Collection A** cohorts by grouping them into 3 distinct categories based on experimental method:
1.  **TMT** (CPTAC/ICPC/CNHPP)
2.  **iTRAQ** (Historical CPTAC)
3.  **Label-Free** (Other matched cohorts)

For each group, we:
- Load the cohort datasets.
- Concatenate them into a unified matrix.
- Filter Features (>50% missing) and Samples (>50% missing).
- Save separate integrated files for Protein and RNA.

In [1]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "1. process_collection_a"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

Figures are saved to C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\05. validation\notebooks\figures_pdf\1. process_collection_a


In [2]:
from pathlib import Path
import sys

# Make the adjacent src/ package importable when this notebook is run from the notebooks folder.
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import src.utils_val as uv

In [3]:
import os
import glob
import pandas as pd
from tqdm import tqdm
import warnings


In [4]:

warnings.simplefilter(action='ignore', category=FutureWarning)

DATA_DIR = r"../data/Collection_A"
OUTPUT_DIR = "../data/processed"
if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

# --- DEFINING COHORT GROUPS ---
COHORT_GROUPS = {
    "TMT": [
        # CPTAC / ICPC / CNHPP
        "colon_2019", "renal_2019", "Endomet_val", "breast2020", "pedbrain_2020",
        "lung_2020_smoke", "lung_val", "lung2021", "headneck_2021", "pancrea2021",
        "brain_2021", "liver_val"
    ],
    "iTRAQ": [
        # Historical CPTAC
        "breast_2016", "ovary_2016", "stomach_2019"
    ],
    "LabelFree": [
        # Other matched cohorts
        "brain", "colon", "prostate", "liver", "lung"
    ]
}

# Initialize storage
group_data = {"TMT": {"prot": [], "rna": []}, "iTRAQ": {"prot": [], "rna": []}, "LabelFree": {"prot": [], "rna": []}}


## 1. Load Data into Groups

In [5]:
subdirs = [f.path for f in os.scandir(DATA_DIR) if f.is_dir()]
print(f"Found {len(subdirs)} subdirectories in {DATA_DIR}")

for subdir in tqdm(subdirs, desc="Processing Cohorts"):
    cohort_name = os.path.basename(subdir)
    
    # Determine Group
    assigned_group = None
    for group_name, cohort_list in COHORT_GROUPS.items():
        if cohort_name in cohort_list:
            assigned_group = group_name
            break
    
    if not assigned_group:
        print(f"[WARNING] Cohort '{cohort_name}' not in defined groups. Skipping.")
        continue

    # Find Files
    prot_files = glob.glob(os.path.join(subdir, "protein_matrix_*.txt"))
    rna_files = glob.glob(os.path.join(subdir, "RNA_matrix_*.txt"))
    
    if prot_files and rna_files:
        try:
            df_p = pd.read_csv(prot_files[0], sep="\t", index_col=0)
            df_r = pd.read_csv(rna_files[0], sep="\t", index_col=0)
            
            # Align Samples
            common_samples = df_p.columns.intersection(df_r.columns)
            
            if len(common_samples) > 0:
                df_p = df_p[common_samples]
                df_r = df_r[common_samples]

                # Rename Samples
                new_names = [f"{cohort_name.upper()}_{i+1}" for i in range(len(common_samples))]
                df_p.columns = new_names
                df_r.columns = new_names
                
                # Store
                group_data[assigned_group]["prot"].append(df_p)
                group_data[assigned_group]["rna"].append(df_r)
        except Exception as e:
            print(f"  -> ERROR in {cohort_name}: {e}")

Found 20 subdirectories in ../data/Collection_A


Processing Cohorts: 100%|██████████| 20/20 [00:11<00:00,  1.69it/s]


## 2. Concatenate, Filter, and Save per Group

In [6]:
# >>> Ensembl -> gene-symbol mapping for RNA *and* protein identifiers
# Several cohorts store proteins (and some RNA) as Ensembl gene IDs (ENSG...), while CCLE uses gene symbols.
# Notebook 2 used to map proteins online with mygene, but notebook 6 reads the files written here, so the
# mapping must happen here. Sources, in order: rows_metadata.csv (offline), then mygene (online, optional).
import re
import numpy as np

MAPPING_FILE = os.path.abspath("../data/Collection_A/brain/rows_metadata.csv")
_ENSG = re.compile(r"^ENSG\d+")


def _strip_version(x):
    return str(x).split(".")[0] if _ENSG.match(str(x)) else str(x)


_offline_map = {}
if os.path.exists(MAPPING_FILE):
    _m = pd.read_csv(MAPPING_FILE).dropna(subset=["ensembl_gene_id", "gene_symbol"])
    _offline_map = {_strip_version(k): str(v) for k, v in zip(_m["ensembl_gene_id"], _m["gene_symbol"])}
print(f"Offline Ensembl->symbol entries (rows_metadata.csv): {len(_offline_map):,}")

mapping_log = []   # (modality, group, ensembl_id, symbol, source)


def map_ensembl_index(df, modality, group, use_online=True):
    """Map an Ensembl-ID index to gene symbols; average rows that collapse onto the same symbol."""
    ids = [_strip_version(x) for x in df.index]
    ensg = sorted({i for i in ids if _ENSG.match(i)})
    if not ensg:
        return df
    mapping = {i: _offline_map[i] for i in ensg if i in _offline_map}
    source = {i: "rows_metadata" for i in mapping}
    missing = [i for i in ensg if i not in mapping]
    if missing and use_online:
        try:
            import mygene
            res = mygene.MyGeneInfo().querymany(missing, scopes="ensembl.gene", fields="symbol",
                                                species="human", as_dataframe=True, verbose=False)
            if "symbol" in res.columns:
                res = res[~res.index.duplicated(keep="first")]["symbol"].dropna()
                for k, v in res.items():
                    mapping[str(k)] = str(v)
                    source[str(k)] = "mygene"
        except Exception as exc:
            print(f"   -> mygene lookup skipped ({type(exc).__name__}: {exc})")
    unmapped = [i for i in ensg if i not in mapping]
    for i in ensg:
        mapping_log.append((modality, group, i, mapping.get(i, ""), source.get(i, "unmapped")))
    print(f"   -> {modality}: {len(ensg):,} Ensembl IDs | mapped {len(ensg) - len(unmapped):,} "
          f"(rows_metadata {sum(v == 'rows_metadata' for v in source.values()):,}, "
          f"mygene {sum(v == 'mygene' for v in source.values()):,}) | unmapped {len(unmapped):,} (kept as ENSG)")
    out = df.copy()
    out.index = [mapping.get(i, i) for i in ids]
    if out.index.duplicated().any():
        n_dup = int(out.index.duplicated().sum())
        out = out.groupby(level=0).mean()
        print(f"   -> {n_dup:,} rows collapsed onto existing symbols (averaged)")
    return out

Offline Ensembl->symbol entries (rows_metadata.csv): 52,376


In [7]:
from src.utils_val import filter_nans

for group_name, dfs in group_data.items():
    print(f"\n=== Processing Group: {group_name} ===")
    
    # Concatenate
    full_protein = pd.concat(dfs["prot"], axis=1, join='outer', sort=True) if dfs["prot"] else pd.DataFrame()
    full_rna = pd.concat(dfs["rna"], axis=1, join='outer', sort=True) if dfs["rna"] else pd.DataFrame()

    # --- MAP Ensembl IDs to gene SYMBOLS (RNA and protein) ---
    if not full_rna.empty:
        full_rna = map_ensembl_index(full_rna, "RNA", group_name)
    if not full_protein.empty:
        full_protein = map_ensembl_index(full_protein, "Protein", group_name)
    # ----------------------------------------------------------
    
    
    if full_protein.empty: 
        print(" [WARNING] No Protein data found.")
        continue
        
    print(f" Initial Shapes -> Protein: {full_protein.shape}, RNA: {full_rna.shape}")

    # Filter
    print(" Filtering Protein...")
    full_protein = filter_nans(full_protein)
    print(" Filtering RNA...")
    full_rna = filter_nans(full_rna)
    
    # Transpose (Samples x Features)
    integrated_protein = full_protein.T
    integrated_rna = full_rna.T
    
    # Remove duplicates
    integrated_protein = integrated_protein.loc[~integrated_protein.index.duplicated(keep='first')]
    integrated_rna = integrated_rna.loc[~integrated_rna.index.duplicated(keep='first')]
    
    print(f" Final Shapes   -> Protein: {integrated_protein.shape}, RNA: {integrated_rna.shape}")

    # --- VIEW HEADS ---
    print(f"Top 5 rows of {group_name} Protein:")
    display(integrated_protein.head())
    print(f"Top 5 rows of {group_name} RNA:")
    display(integrated_rna.head())
    # ------------------
    

    # Save
    p_out = os.path.join(OUTPUT_DIR, f"Collection_A_{group_name}_Protein.csv")
    r_out = os.path.join(OUTPUT_DIR, f"Collection_A_{group_name}_RNA.csv")
    
    integrated_protein.to_csv(p_out)
    integrated_rna.to_csv(r_out)
    print(f" Saved to {p_out} and {r_out}")
    ensg_left = int(pd.Index(integrated_protein.columns.astype(str)).str.match(r"^ENSG\d+").sum())
    print(f" Protein columns still Ensembl IDs: {ensg_left:,} of {integrated_protein.shape[1]:,}")


=== Processing Group: TMT ===


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


   -> RNA: 33,554 Ensembl IDs | mapped 30,714 (rows_metadata 28,626, mygene 2,088) | unmapped 2,840 (kept as ENSG)


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


   -> 920 rows collapsed onto existing symbols (averaged)
   -> Protein: 11,259 Ensembl IDs | mapped 11,255 (rows_metadata 11,105, mygene 150) | unmapped 4 (kept as ENSG)
   -> 4 rows collapsed onto existing symbols (averaged)
 Initial Shapes -> Protein: (11255, 1924), RNA: (32634, 1924)
 Filtering Protein...
   -> Dropped 4371 Features & 0 Samples
 Filtering RNA...
   -> Dropped 17625 Features & 0 Samples
 Final Shapes   -> Protein: (1924, 6884), RNA: (1924, 15009)
Top 5 rows of TMT Protein:


,A1BG,A2LD1,A2M,AAAS,AACS,AAGAB,AAK1,AAMP,AARS,AARS2,...,ZNF830,ZNFX1,ZNHIT2,ZNRF2,ZRANB2,ZSCAN18,ZW10,ZYG11B,ZYX,ZZEF1
BRAIN_2021_1,0.078,-0.437,0.487,-0.254,-0.144,-0.467,-0.090,0.330,-0.318,0.336,...,-0.031,-0.151,-0.339,-0.469,0.160,0.660,-0.106,0.079,-0.785,-0.488
BRAIN_2021_2,-0.146,0.391,0.799,0.184,-0.471,0.245,-0.610,-0.087,0.134,-0.316,...,0.305,0.136,-0.057,0.010,0.077,-1.272,-0.213,0.049,0.138,-0.290
BRAIN_2021_3,0.822,0.388,1.096,-0.094,-0.106,-0.248,0.014,-0.638,-0.367,-0.158,...,-0.003,0.418,-0.724,-0.437,-0.795,-0.165,-0.307,-0.172,0.012,-0.132
BRAIN_2021_4,-0.065,0.753,0.129,0.048,-0.118,0.304,0.322,-0.363,0.595,0.181,...,0.054,-0.101,-0.184,-0.259,-1.056,-0.600,0.464,0.178,-0.720,-0.150
BRAIN_2021_5,-0.764,-0.066,-1.032,-0.217,-0.696,-0.052,0.345,0.069,-0.325,0.309,...,0.176,0.123,-0.046,-0.644,0.160,-0.752,0.011,0.113,1.005,-0.230


... (15 outputs left out)

,A1BG,A2M,AARS,AARS2,AARSD1,AASDHPPT,ABAT,ABCB7,ABCD3,ABCE1,...,YWHAQ,YWHAZ,ZC3H15,ZC3HAV1,ZFR,ZMPSTE24,ZNF207,ZW10,ZYX,ZZEF1
BRAIN_1,6.98,9.435,9.745,7.422,8.352,8.581,10.012,6.823,8.329,8.591,...,10.229,10.932,6.982,NaN,7.000,7.935,7.035,7.435,8.098,7.567
BRAIN_2,6.98,9.790,9.747,7.852,8.444,8.584,10.301,7.927,8.948,8.481,...,10.416,10.698,6.982,NaN,8.187,7.665,8.568,7.599,8.641,7.569
BRAIN_3,6.98,9.794,9.816,7.706,8.243,8.695,9.891,7.827,8.781,8.706,...,10.215,10.946,7.768,NaN,7.000,7.910,7.035,7.830,8.226,7.615
BRAIN_4,6.98,9.633,9.712,7.676,8.294,8.641,10.365,7.727,8.860,8.228,...,10.155,10.849,6.982,NaN,7.000,8.016,7.035,7.774,8.294,7.639
BRAIN_5,6.98,9.631,9.790,7.770,8.010,9.113,10.262,7.893,8.639,8.248,...,10.104,10.918,7.737,NaN,7.000,8.031,7.817,7.912,8.098,7.641


Top 5 rows of LabelFree RNA:


,A1BG,A1BG-AS1,A1CF,A2LD1,A2M,A4GALT,AAAS,AACS,AADAT,AAGAB,...,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11B,ZYX,ZZEF1,ZZZ3
BRAIN_1,12.893,NaN,NaN,0.826,21.054,NaN,4.548,2.540,3.030,10.040,...,2.241,2.650,2.489,0.805,2.664,0.600,11.526,11.135,1.012,1.781
BRAIN_2,8.307,NaN,NaN,0.939,20.753,NaN,9.086,1.035,0.474,10.601,...,3.313,2.422,0.679,1.019,2.700,1.964,10.713,35.617,2.349,1.895
BRAIN_3,15.115,NaN,NaN,0.770,18.468,NaN,3.590,2.796,2.194,8.735,...,1.143,1.566,1.440,0.422,0.939,0.278,6.985,8.945,0.497,1.029
BRAIN_4,2.053,NaN,NaN,0.751,35.283,NaN,4.274,2.869,1.592,17.248,...,3.184,1.881,3.975,0.663,3.019,0.717,15.983,10.402,2.042,2.123
BRAIN_5,6.533,NaN,NaN,1.110,29.215,NaN,5.186,1.586,2.978,10.827,...,2.893,1.973,2.196,0.839,2.132,0.863,6.988,6.625,1.148,1.810


 Saved to ../data/processed\Collection_A_LabelFree_Protein.csv and ../data/processed\Collection_A_LabelFree_RNA.csv
 Protein columns still Ensembl IDs: 0 of 2,565


In [8]:
# Save the identifier mapping that was used (reproducibility)
mapping_table = pd.DataFrame(mapping_log, columns=["modality", "group", "ensembl_id", "symbol", "source"])
mapping_table.to_csv(os.path.join(OUTPUT_DIR, "ensembl_to_symbol_mapping_used.csv"), index=False)
display(mapping_table.groupby(["modality", "group", "source"]).size().unstack(fill_value=0))

source              mygene  rows_metadata  unmapped
modality group                                     
Protein  LabelFree      55           5710         2
         TMT           150          11105         4
         iTRAQ          84           7483         2
RNA      LabelFree    1511          22880       838
         TMT          2088          28626      2840
         iTRAQ         690          17527       642